# Riscrittura codice Cleaning_2 sostituendo l'excel 

Caricamento librerie e file

In [30]:
from __future__ import annotations
import re
import numpy as np
import pandas as pd

import config
from config import DatasetConfig, ADNIMERGE

ADNIMERGE.source = "ADNIMERGE_cleaned_01.csv"
OUTPUT_FILE = "ADNIMERGE_cleaned_02.csv"
CUTOFFS_FILE = "cutoffs.json"

Funzione da richiamare per salvare le modifiche su ADNIMERGE_cleaned_02.csv

In [3]:
def save_dataset(df, path):
   df.to_csv(path, index=False)
   print(f"Salvato: {path}  (shape: {df.shape})")

## Rimozione colonne con troppi valori non validi per l'analizi
Funzione di pulizia colonne. Calcola, per ogni colonna, la percentuale di valori validi e scarta quelle sotto la soglia definita in config.MISSING_KEEP_THRESHOLD.

In [9]:
def remove_param_few_subjects(df, threshold=config.MISSING_KEEP_THRESHOLD):
    df = df.copy()
    valid_ratio = df.notna().mean()
    dropped = valid_ratio[valid_ratio < threshold].index.tolist()
    df = df.drop(columns=dropped)
    return df, dropped

Esecuzione dello step. Legge il file originale, applica la pulizia, stampa quali colonne sono state scartate e la variazione di shape, poi salva il risultato con save_dataset().

In [10]:
df_raw = pd.read_csv(ADNIMERGE.source)

df_cleaned, dropped_columns = remove_param_few_subjects(df_raw)
 
print(f"[STEP 1] Colonne scartate ({len(dropped_columns)}): {dropped_columns}")
print(f"[STEP 1] Shape: {df_raw.shape} -> {df_cleaned.shape}")
 
save_dataset(df_cleaned, OUTPUT_FILE)  

C:\Users\ifabb\AppData\Local\Temp\ipykernel_2188\914369842.py:1: DtypeWarning: Columns (0: TTAU_CSF, 1: TAU_bl, 2: PTAU_bl) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv(ADNIMERGE.source)


[STEP 1] Colonne scartate (46): ['FDG', 'PIB', 'AV45', 'FBB', 'AB42_CSF', 'TTAU_CSF', 'PT181_CSF', 'DIGITSCOR', 'MOCA', 'EcogPtMem', 'EcogPtLang', 'EcogPtVisspat', 'EcogPtPlan', 'EcogPtOrgan', 'EcogPtDivatt', 'EcogPtTotal', 'EcogSPMem', 'EcogSPLang', 'EcogSPVisspat', 'EcogSPPlan', 'EcogSPOrgan', 'EcogSPDivatt', 'EcogSPTotal', 'FLDSTRENG', 'DIGITSCOR_bl', 'MOCA_bl', 'EcogPtMem_bl', 'EcogPtLang_bl', 'EcogPtVisspat_bl', 'EcogPtPlan_bl', 'EcogPtOrgan_bl', 'EcogPtDivatt_bl', 'EcogPtTotal_bl', 'EcogSPMem_bl', 'EcogSPLang_bl', 'EcogSPVisspat_bl', 'EcogSPPlan_bl', 'EcogSPOrgan_bl', 'EcogSPDivatt_bl', 'EcogSPTotal_bl', 'ABETA_bl', 'TAU_bl', 'PTAU_bl', 'PIB_bl', 'AV45_bl', 'FBB_bl']
[STEP 1] Shape: (11458, 118) -> (11458, 72)
Salvato: ADNIMERGE_cleaned_02.csv  (shape: (11458, 72))


Verifica di coerenza. Rilegge il file appena scritto e controlla che corrisponda davvero a df_cleaned, per evitare di proseguire con un file non aggiornato (il problema riscontrato in precedenza).

In [6]:
check_step1 = pd.read_csv(OUTPUT_FILE)
assert check_step1.shape == df_cleaned.shape, "[STEP 1] File salvato NON corrisponde a df_cleaned!"
print(f"[STEP 1] Verifica OK: '{OUTPUT_FILE}' ha shape {check_step1.shape}")

[STEP 1] Verifica OK: 'ADNIMERGE_cleaned_02.csv' ha shape (11458, 72)


Controlla le dimensioni prima/dopo, per avere conferma numerica:

## Conversione in dummy delle colonne 'GENDER', 'MARRY', 'ETHNICITY', 'RACE', 'DX'
Configurazione dello step. Definisce input/output (il file appena prodotto dallo STEP 1) e l'elenco delle variabili categoriche da convertire.

In [12]:
DUMMY_INPUT = OUTPUT_FILE             # <-- usa direttamente l'output dello step 1
DUMMY_OUTPUT = "ADNIMERGE_cleaned_02.csv"

REF_LIST = ['GENDER', 'MARRY', 'ETHNICITY', 'RACE', 'DX']

Funzione di conversione in dummy. Individua tra ref_list le colonne presenti nel dataframe e le trasforma in variabili binarie con pd.get_dummies.

In [13]:
def classes_to_dummies(df, ref_list):
    df = df.copy()
    to_dummy_list = [c for c in ref_list if c in df.columns]
    if to_dummy_list:
        df = pd.get_dummies(df, columns=to_dummy_list, dtype=int)
    return df, to_dummy_list

Funzione di conteggio dummy. Conta quante colonne dummy sono state create in totale e quante per ciascuna variabile originale.

In [14]:
def count_dummy_columns(original_df, final_df, converted_columns):
    new_dummy_columns = [c for c in final_df.columns if c not in original_df.columns]
    per_column_count = {
        col: len([c for c in new_dummy_columns if c.startswith(col + "_")])
        for col in converted_columns
    }
    return len(new_dummy_columns), per_column_count

Controllo di sicurezza pre-esecuzione. Legge il file d'ingresso e verifica che abbia già il numero di colonne atteso dallo STEP 1 (72), bloccando l'esecuzione con un messaggio chiaro se non è così.

In [15]:
df_step2 = pd.read_csv(DUMMY_INPUT)

In [16]:
assert df_step2.shape[1] == df_cleaned.shape[1], (
    f"[STEP 2] Attenzione: '{DUMMY_INPUT}' ha {df_step2.shape[1]} colonne, "
    f"attese {df_cleaned.shape[1]}. Rieseguire lo STEP 1 prima di procedere."
)

Esecuzione e salvataggio. Applica la conversione in dummy e salva il risultato con save_dataset().

In [17]:
final_df, converted_columns = classes_to_dummies(df_step2, ref_list=REF_LIST)

save_dataset(final_df, DUMMY_OUTPUT)

Salvato: ADNIMERGE_cleaned_02.csv  (shape: (11458, 84))


### Report finale. Riepiloga colonne convertite, variazione di shape e conteggio dettagliato delle dummy create per ciascuna variabile.

In [18]:
total_dummy_count, per_column_count = count_dummy_columns(df_step2, final_df, converted_columns)
print(f"[STEP 2] Colonne convertite in dummy: {converted_columns}")
print(f"[STEP 2] Shape: {df_step2.shape} -> {final_df.shape}")
print(f"[STEP 2] Totale colonne dummy create: {total_dummy_count}")
for col, n in per_column_count.items():
    print(f"  '{col}' -> {n} colonne dummy")

[STEP 2] Colonne convertite in dummy: ['GENDER', 'MARRY', 'ETHNICITY', 'RACE', 'DX']
[STEP 2] Shape: (11458, 72) -> (11458, 84)
[STEP 2] Totale colonne dummy create: 17
  'GENDER' -> 2 colonne dummy
  'MARRY' -> 4 colonne dummy
  'ETHNICITY' -> 2 colonne dummy
  'RACE' -> 6 colonne dummy
  'DX' -> 3 colonne dummy


Controllo modifica andata a buon fine.

In [ ]:
pd.read_csv("ADNIMERGE_cleaned_02.csv").head(5)

,RID,COLPROT,ORIGPROT,PTID,SITE,VISCODE,EXAMDATE,DX_bl,AGE_bl,EDUCATION,...,ETHNICITY_1.0,RACE_0.0,RACE_1.0,RACE_2.0,RACE_3.0,RACE_4.0,RACE_5.0,DX_0,DX_1,DX_2
0,2,ADNI1,ADNI1,011_S_0002,11,bl,2005-09-08,CN,74.3,16,...,0,0,0,0,0,0,1,1,0,0
1,2,ADNI1,ADNI1,011_S_0002,11,m06,2006-03-06,CN,74.3,16,...,0,0,0,0,0,0,1,1,0,0
2,2,ADNI1,ADNI1,011_S_0002,11,m36,2008-08-27,CN,74.3,16,...,0,0,0,0,0,0,1,1,0,0
3,2,ADNIGO,ADNI1,011_S_0002,11,m60,2010-09-22,CN,74.3,16,...,0,0,0,0,0,0,1,1,0,0
4,2,ADNI2,ADNI1,011_S_0002,11,m72,2011-09-19,CN,74.3,16,...,0,0,0,0,0,0,1,1,0,0


Controllo solo le colonne aggiunt dummy.

In [ ]:
final_df[[c for c in final_df.columns if c not in df_step2.columns]].tail(5)

,GENDER_0,GENDER_1,MARRY_0.0,MARRY_1.0,MARRY_2.0,MARRY_3.0,ETHNICITY_0.0,ETHNICITY_1.0,RACE_0.0,RACE_1.0,RACE_2.0,RACE_3.0,RACE_4.0,RACE_5.0,DX_0,DX_1,DX_2
11453,0,1,0,1,0,0,0,1,1,0,0,0,0,0,0,1,0
11454,1,0,0,0,1,0,1,0,0,0,0,0,1,0,1,0,0
11455,1,0,1,0,0,0,1,0,0,0,0,0,1,0,0,1,0
11456,0,1,0,1,0,0,1,0,0,0,0,0,0,1,0,1,0
11457,1,0,0,1,0,0,0,1,1,0,0,0,0,0,1,0,0


## Rimozione righe in cui tutte le colonne chiave sono vuote

Colonne chiave: misure volumetriche cerebrali fondamentali

In [19]:
KEY_COLUMNS = ['Ventricles', 'Hippocampus', 'WholeBrain', 'ICV']

Funzione di filtro sulle righe

In [ ]:
def drop_if_all_none(df, key_columns):
    df = df.copy()
    cols_present = [c for c in key_columns if c in df.columns]
    mask_all_none = df[cols_present].isna().all(axis=1)
    dropped_rows = int(mask_all_none.sum())
    df = df[~mask_all_none]
    return df, dropped_rows

Caricamento e applicazione del filtro

In [23]:
df_step3 = pd.read_csv(OUTPUT_FILE)
df_final, dropped_rows = drop_if_all_none(df_step3, KEY_COLUMNS)

Controllo

In [24]:
print(f"[STEP 3] Colonne chiave verificate: {KEY_COLUMNS}")
print(f"[STEP 3] Righe scartate (tutte le colonne chiave nulle): {dropped_rows}")
print(f"[STEP 3] Shape: {df_step3.shape} -> {df_final.shape}")

[STEP 3] Colonne chiave verificate: ['Ventricles', 'Hippocampus', 'WholeBrain', 'ICV']
[STEP 3] Righe scartate (tutte le colonne chiave nulle): 2152
[STEP 3] Shape: (11458, 84) -> (9306, 84)


In [25]:
save_dataset(df_final, OUTPUT_FILE)

Salvato: ADNIMERGE_cleaned_02.csv  (shape: (9306, 84))


### Statistiche sui soggetti: totale e visite multiple

In [37]:
def count_unique_subjects(df, id_column='RID'):
    """Conta il numero totale di soggetti unici nel dataset."""
    tot_sub = len(df[id_column].unique())
    return tot_sub

In [38]:
def count_multiple_visits(df, id_column='RID'):
    """Conta quanti soggetti hanno più di una visita."""
    multiple_visits = (df[id_column].value_counts() > 1).sum()
    return multiple_visits

In [39]:
final_df = pd.read_csv(OUTPUT_FILE)


C:\Users\ifabb\AppData\Local\Temp\ipykernel_2188\4233954804.py:1: DtypeWarning: Columns (0: FLDSTRENG_bl) have mixed types. Specify dtype option on import or set low_memory=False.
  final_df = pd.read_csv(OUTPUT_FILE)


In [40]:
tot_sub = count_unique_subjects(final_df)
print('totale subject:', tot_sub)

totale subject: 2348


In [41]:
multiple_visits = count_multiple_visits(final_df)
print('subjects with multiple visits: ', multiple_visits)

subjects with multiple visits:  1930


## Rialineamento types delle colonne

Funzione di riallineamento che controllo ogni collonna

In [42]:
def realign_column_types(df):
    """Ispeziona ogni colonna e la riallinea al tipo corretto (int, float o testo)."""
    df = df.copy()
    type_report = {}

    for col in df.columns:
        original_dtype = str(df[col].dtype)

        # Prova la conversione numerica; se fallisce, la colonna resta testo
        converted = pd.to_numeric(df[col], errors='coerce')
        is_fully_numeric = converted.notna().sum() == df[col].notna().sum()

        if is_fully_numeric:
            if (converted.dropna() % 1 == 0).all():
                df[col] = converted.astype('Int64')   # int nullable, gestisce i NaN
                new_dtype = 'Int64'
            else:
                df[col] = converted.astype('float64')
                new_dtype = 'float64'
        else:
            df[col] = df[col].astype(str).where(df[col].notna(), np.nan)
            new_dtype = 'str'

        if original_dtype != new_dtype:
            type_report[col] = (original_dtype, new_dtype)

    return df, type_report


In [43]:
final_df = pd.read_csv(OUTPUT_FILE)
final_df, type_report = realign_column_types(final_df)

C:\Users\ifabb\AppData\Local\Temp\ipykernel_2188\438724613.py:1: DtypeWarning: Columns (0: FLDSTRENG_bl) have mixed types. Specify dtype option on import or set low_memory=False.
  final_df = pd.read_csv(OUTPUT_FILE)


Controllo

In [45]:
print(f"[STEP4] Colonne con tipo riallineato: {len(type_report)}")
for col, (old, new) in type_report.items():
    print(f"  '{col}': {old} -> {new}")

[STEP4] Colonne con tipo riallineato: 51
  'RID': int64 -> Int64
  'SITE': int64 -> Int64
  'EDUCATION': int64 -> Int64
  'APOE4': float64 -> Int64
  'ADASQ4': float64 -> Int64
  'MMSE': float64 -> Int64
  'RAVLT_immediate': float64 -> Int64
  'RAVLT_learning': float64 -> Int64
  'RAVLT_forgetting': float64 -> Int64
  'LDELTOTAL': float64 -> Int64
  'FAQ': float64 -> Int64
  'IMAGEUID': float64 -> Int64
  'WholeBrain': float64 -> Int64
  'Entorhinal': float64 -> Int64
  'Fusiform': float64 -> Int64
  'MidTemp': float64 -> Int64
  'ICV': float64 -> Int64
  'ADASQ4_bl': float64 -> Int64
  'MMSE_bl': float64 -> Int64
  'RAVLT_immediate_bl': float64 -> Int64
  'RAVLT_learning_bl': float64 -> Int64
  'RAVLT_forgetting_bl': float64 -> Int64
  'LDELTOTAL_BL': float64 -> Int64
  'TRABSCOR_bl': float64 -> Int64
  'FAQ_bl': float64 -> Int64
  'IMAGEUID_bl': float64 -> Int64
  'WholeBrain_bl': float64 -> Int64
  'Entorhinal_bl': float64 -> Int64
  'Fusiform_bl': float64 -> Int64
  'MidTemp_bl': f

In [46]:
save_dataset(final_df, OUTPUT_FILE)

Salvato: ADNIMERGE_cleaned_02.csv  (shape: (9306, 84))
